In [1]:
import sys
import json
import platform
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import torch

OUTPUT_DIR = Path("/kaggle/working/HealthVer_Experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

environment = {
    "python": sys.version.split()[0],
    "platform": platform.system(),
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "packages": {},
    "gpus": [],
}

for package in [
    "torch", "transformers", "datasets", "peft",
    "accelerate", "huggingface-hub", "scikit-learn", "pandas",
]:
    try:
        environment["packages"][package] = version(package)
    except PackageNotFoundError:
        environment["packages"][package] = "NOT INSTALLED"

if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_properties(index)
        free, total = torch.cuda.mem_get_info(index)

        environment["gpus"].append({
            "index": index,
            "name": gpu.name,
            "total_gib": round(total / 1024**3, 2),
            "free_gib": round(free / 1024**3, 2),
            "compute_capability": f"{gpu.major}.{gpu.minor}",
        })

print(json.dumps(environment, indent=2))

(OUTPUT_DIR / "environment.json").write_text(
    json.dumps(environment, indent=2)
)

assert environment["cuda_available"], (
    "Enable a GPU in Kaggle notebook settings."
)

{
  "python": "3.13.15",
  "platform": "Linux",
  "torch_cuda_build": "12.8",
  "cuda_available": true,
  "packages": {
    "torch": "2.11.0+cu128",
    "transformers": "5.16.1",
    "datasets": "4.8.5",
    "peft": "0.20.0",
    "accelerate": "1.14.0",
    "huggingface-hub": "1.29.0",
    "scikit-learn": "1.6.1",
    "pandas": "2.3.3"
  },
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "total_gib": 14.56,
      "free_gib": 14.46,
      "compute_capability": "7.5"
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "total_gib": 14.56,
      "free_gib": 14.46,
      "compute_capability": "7.5"
    }
  ]
}


In [2]:
import json
import hashlib
import urllib.request
from pathlib import Path

import pandas as pd
from datasets import Dataset, DatasetDict

DATASET_REVISION = "b20ac99ceed62f5264a319fa25a854df1668d85b"
SOURCE_URL = "https://github.com/sarrouti/HealthVer"

DATA_DIR = OUTPUT_DIR / "original_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

split_files = {
    "train": "healthver_train.csv",
    "validation": "healthver_dev.csv",
    "test": "healthver_test.csv",
}
expected_sizes = {
    "train": 10590,
    "validation": 1917,
    "test": 1823,
}

raw_ds = DatasetDict()
file_metadata = {}

for split, filename in split_files.items():
    url = (
        "https://raw.githubusercontent.com/sarrouti/HealthVer/"
        f"{DATASET_REVISION}/data/{filename}"
    )

    # Separate filenames by revision to avoid mixing cached versions.
    path = DATA_DIR / f"{DATASET_REVISION}_{filename}"

    if not path.exists():
        with urllib.request.urlopen(url, timeout=60) as response:
            content = response.read()

        temporary_path = path.with_suffix(".tmp")
        temporary_path.write_bytes(content)
        temporary_path.replace(path)

    # Preserve IDs as strings; do not silently convert text to missing values.
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)

    assert {"id", "claim", "evidence", "label"}.issubset(frame.columns)
    assert len(frame) == expected_sizes[split], (
        f"Unexpected {split} size: {len(frame)}"
    )

    raw_ds[split] = Dataset.from_pandas(frame, preserve_index=False)

    file_metadata[split] = {
        "url": url,
        "rows": len(frame),
        "columns": frame.columns.tolist(),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }

metadata = {
    "dataset": "Original HealthVer",
    "source": SOURCE_URL,
    "revision": DATASET_REVISION,
    "files": file_metadata,
    "cleaning": "none",
    "resplitting": False,
}
(OUTPUT_DIR / "dataset_metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("Dataset: Original HealthVer")
print("Pinned revision:", DATASET_REVISION)

for split, dataset in raw_ds.items():
    print(f"{split}: {len(dataset):,} rows")

print("\nColumns:", raw_ds["train"].column_names)
print("Features:", raw_ds["train"].features)
print("\nOriginal splits preserved. No cleaning or sampling applied.")

Dataset: Original HealthVer
Pinned revision: b20ac99ceed62f5264a319fa25a854df1668d85b
train: 10,590 rows
validation: 1,917 rows
test: 1,823 rows

Columns: ['id', 'evidence', 'claim', 'label', 'topic_ip', 'question']
Features: {'id': Value('string'), 'evidence': Value('string'), 'claim': Value('string'), 'label': Value('string'), 'topic_ip': Value('string'), 'question': Value('string')}

Original splits preserved. No cleaning or sampling applied.


In [3]:
import hashlib
import json
from itertools import combinations

import pandas as pd
from IPython.display import display

def text_hash(*texts):
    # Whitespace normalization for auditing only.
    normalized = [" ".join(text.split()) for text in texts]
    return hashlib.sha256(
        json.dumps(normalized, ensure_ascii=False).encode("utf-8")
    ).hexdigest()

audit_rows = []
class_rows = []
split_keys = {}

for split, dataset in raw_ds.items():
    frame = dataset.to_pandas()

    pair_hashes = pd.Series([
        text_hash(claim, evidence)
        for claim, evidence in zip(frame["claim"], frame["evidence"])
    ])
    claim_hashes = frame["claim"].map(text_hash)
    evidence_hashes = frame["evidence"].map(text_hash)

    summary = {
        "split": split,
        "rows": len(frame),
        "duplicate_id_extra_rows": int(frame["id"].duplicated().sum()),
        "duplicate_pair_extra_rows": int(pair_hashes.duplicated().sum()),
        "unique_claims": int(claim_hashes.nunique()),
    }

    for column in ["id", "claim", "evidence", "label"]:
        values = frame[column].astype("string")
        blank = values.isna() | values.str.strip().eq("").fillna(False)
        summary[f"blank_{column}"] = int(blank.sum())

    if split != "test":
        pair_labels = pd.DataFrame({
            "pair_hash": pair_hashes,
            "label": frame["label"],
        })
        summary["conflicting_pair_groups"] = int(
            pair_labels.groupby("pair_hash")["label"].nunique().gt(1).sum()
        )

        counts = frame["label"].value_counts()
        for label, count in counts.items():
            class_rows.append({
                "split": split,
                "label": label,
                "count": int(count),
                "percentage": round(100 * count / len(frame), 2),
            })

    audit_rows.append(summary)
    split_keys[split] = {
        "id": set(frame["id"]),
        "claim": set(claim_hashes),
        "evidence": set(evidence_hashes),
        "input_pair": set(pair_hashes),
    }

overlap_rows = []
for split_a, split_b in combinations(raw_ds.keys(), 2):
    overlap_rows.append({
        "split_a": split_a,
        "split_b": split_b,
        **{
            f"shared_{key}": len(
                split_keys[split_a][key] & split_keys[split_b][key]
            )
            for key in split_keys[split_a]
        },
    })

tables = {
    "class_distribution": pd.DataFrame(class_rows),
    "dataset_audit": pd.DataFrame(audit_rows),
    "cross_split_overlap": pd.DataFrame(overlap_rows),
}

for name, table in tables.items():
    print(f"\n{name.upper()}")
    display(table)
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

print("\nONE TRAINING EXAMPLE PER CLASS")
train_frame = raw_ds["train"].to_pandas()

for label, group in train_frame.groupby("label", sort=True):
    example = group.iloc[0]
    print(f"\nLabel: {label} | ID: {example['id']}")
    print("Claim:", example["claim"])
    print("Evidence:", example["evidence"])

print("\nAudit saved. Original data unchanged.")


CLASS_DISTRIBUTION


,split,label,count,percentage
0,train,Neutral,4397,41.52
1,train,Supports,3782,35.71
2,train,Refutes,2411,22.77
3,validation,Neutral,993,51.80
4,validation,Supports,533,27.80
5,validation,Refutes,391,20.40



DATASET_AUDIT


,split,rows,duplicate_id_extra_rows,duplicate_pair_extra_rows,unique_claims,blank_id,blank_claim,blank_evidence,blank_label,conflicting_pair_groups
0,train,10590,0,700,1392,0,0,0,0,0.0
1,validation,1917,0,198,230,0,0,0,0,0.0
2,test,1823,0,129,230,0,0,0,0,NaN



CROSS_SPLIT_OVERLAP


,split_a,split_b,shared_id,shared_claim,shared_evidence,shared_input_pair
0,train,validation,0,0,465,0
1,train,test,0,1,451,0
2,validation,test,0,0,374,0



ONE TRAINING EXAMPLE PER CLASS

Label: Neutral | ID: 6321
Claim: The good news. Properly disinfecting our homes and commonly touched objects helps prevent the spread of all contagious diseases, including COVID-19.
Evidence: As discussed in this review, till effective vaccines and treatments emerge, it is important to understand the scientific rationale of pandemic-mitigation strategies such as wearing facemasks and social distancing, and implement them.

Label: Refutes | ID: 2868
Claim: Results on the use of hydroxychloroquine as a treatment for Covid-19 have shown no significant differences in health outcomes between the control group and patients who received the experimental drug. 
Evidence: HCQ should be prescribed as a part of treatment for critically ill COVID-19 patients, with possible outcome of saving lives.

Label: Supports | ID: 4393
Claim: If you decide to engage in public activities, continue to protect yourself by practicing everyday preventive actions.; Keep these items

In [4]:
import json
import numpy as np
import pandas as pd

from huggingface_hub import HfApi
from transformers import AutoTokenizer
from tqdm.auto import tqdm

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG_PATH = OUTPUT_DIR / "input_config.json"

# Reuse the pinned revision when rerunning this cell.
if CONFIG_PATH.exists():
    previous_config = json.loads(CONFIG_PATH.read_text())
    assert previous_config["model"] == MODEL_ID
    MODEL_REVISION = previous_config["model_revision"]
else:
    MODEL_REVISION = HfApi().model_info(MODEL_ID).sha

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

LABELS = ["Supports", "Refutes", "Neutral"]

SYSTEM_PROMPT = (
    "Classify the claim using only the provided evidence. "
    "Supports: the evidence supports the claim. "
    "Refutes: the evidence contradicts the claim. "
    "Neutral: the evidence does not establish support or contradiction. "
    "Return exactly one label: Supports, Refutes, or Neutral."
)

def make_messages(claim, evidence):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Claim: {claim}\n\nEvidence: {evidence}",
        },
    ]

length_records = []

for split in ["train", "validation"]:
    dataset = raw_ds[split]

    for offset in tqdm(
        range(0, len(dataset), 256),
        desc=f"{split} token lengths",
    ):
        batch = dataset[offset:offset + 256]
        assert set(batch["label"]).issubset(LABELS)

        conversations = [
            make_messages(claim, evidence)
            for claim, evidence in zip(batch["claim"], batch["evidence"])
        ]

        prompts = tokenizer.apply_chat_template(
            conversations,
            tokenize=False,
            add_generation_prompt=True,
        )
        completed = tokenizer.apply_chat_template(
            [
                messages + [{"role": "assistant", "content": label}]
                for messages, label in zip(conversations, batch["label"])
            ],
            tokenize=False,
            add_generation_prompt=False,
        )

        prompt_ids = tokenizer(
            prompts, add_special_tokens=False, truncation=False
        )["input_ids"]
        full_ids = tokenizer(
            completed, add_special_tokens=False, truncation=False
        )["input_ids"]

        for uid, prompt, full in zip(batch["id"], prompt_ids, full_ids):
            assert full[:len(prompt)] == prompt
            assert len(full) > len(prompt)

            length_records.append({
                "split": split,
                "id": uid,
                "prompt_tokens": len(prompt),
                "training_tokens": len(full),
            })

length_df = pd.DataFrame(length_records)

print("\nCOMPLETE TRAINING SEQUENCE LENGTHS")
display(
    length_df.groupby("split")["training_tokens"]
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99])
    .round(1)
)

print("\nPERCENTAGE EXCEEDING EACH LIMIT")
display(pd.DataFrame([
    {
        "split": split,
        **{
            f">{limit} (%)": round(
                100 * (group["training_tokens"] > limit).mean(), 2
            )
            for limit in [256, 384, 512, 1024]
        },
    }
    for split, group in length_df.groupby("split")
]))

length_df.to_csv(OUTPUT_DIR / "token_lengths.csv", index=False)

CONFIG_PATH.write_text(json.dumps({
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "system_prompt": SYSTEM_PROMPT,
    "labels": LABELS,
    "dataset_revision": DATASET_REVISION,
    "truncation": False,
}, indent=2))

print("\nTokenizer ready. No truncation applied. Test split untouched.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

train token lengths:   0%|          | 0/42 [00:00<?, ?it/s]

... (2 outputs left out)

,count,mean,std,min,50%,90%,95%,99%,max
split,,,,,,,,,
train,10590.0,143.4,34.1,90.0,136.0,185.0,203.0,276.0,328.0
validation,1917.0,145.2,36.4,90.0,139.0,191.0,214.0,277.0,322.0



PERCENTAGE EXCEEDING EACH LIMIT


,split,>256 (%),>384 (%),>512 (%),>1024 (%)
0,train,2.20,0.0,0.0,0.0
1,validation,2.76,0.0,0.0,0.0



Tokenizer ready. No truncation applied. Test split untouched.


In [5]:
from datasets import DatasetDict

MAX_SEQUENCE_LENGTH = 384

def tokenize_batch(batch):
    conversations = [
        make_messages(claim, evidence)
        for claim, evidence in zip(batch["claim"], batch["evidence"])
    ]

    prompt_texts = tokenizer.apply_chat_template(
        conversations,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_texts = tokenizer.apply_chat_template(
        [
            messages + [{"role": "assistant", "content": label}]
            for messages, label in zip(conversations, batch["label"])
        ],
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(
        prompt_texts,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    encoded = tokenizer(
        full_texts,
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=True,
    )

    labels = []
    for prompt, full in zip(prompt_ids, encoded["input_ids"]):
        assert full[:len(prompt)] == prompt
        assert len(prompt) < len(full) <= MAX_SEQUENCE_LENGTH

        labels.append(
            [-100] * len(prompt) + full[len(prompt):]
        )

    encoded["labels"] = labels
    encoded["id"] = batch["id"]
    encoded["sequence_length"] = [
        len(ids) for ids in encoded["input_ids"]
    ]
    return encoded

tokenized_ds = DatasetDict()

for split in ["train", "validation"]:
    tokenized_ds[split] = raw_ds[split].map(
        tokenize_batch,
        batched=True,
        batch_size=256,
        remove_columns=raw_ds[split].column_names,
        desc=f"Preparing {split}",
    )

    assert len(tokenized_ds[split]) == len(raw_ds[split])

    tokenized_ds[split].save_to_disk(
        str(OUTPUT_DIR / f"tokenized_{split}")
    )

# Confirm masking and target text.
example = tokenized_ds["train"][0]
target_ids = [token for token in example["labels"] if token != -100]

print("\nTARGET CHECK")
print("Original label:", raw_ds["train"][0]["label"])
print("Loss target:", repr(
    tokenizer.decode(target_ids, skip_special_tokens=False)
))
print("Masked prompt tokens:", example["labels"].count(-100))

for split, dataset in tokenized_ds.items():
    print(
        f"{split}: {len(dataset):,} rows | "
        f"maximum length: {max(dataset['sequence_length'])}"
    )

config = json.loads(CONFIG_PATH.read_text())
config.update({
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "padding": "dynamic",
    "loss": "assistant_answer_only",
})
CONFIG_PATH.write_text(json.dumps(config, indent=2))

print("\nSaved. All rows preserved; no truncation; test untouched.")

Preparing train:   0%|          | 0/10590 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10590 [00:00<?, ? examples/s]

Preparing validation:   0%|          | 0/1917 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1917 [00:00<?, ? examples/s]


TARGET CHECK
Original label: Refutes
Loss target: 'Refutes<|im_end|>\n'
Masked prompt tokens: 134
train: 10,590 rows | maximum length: 328
validation: 1,917 rows | maximum length: 322

Saved. All rows preserved; no truncation; test untouched.


In [6]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [7]:
import gc
import math
import time
import numpy as np
import torch

from torch.nn.utils.rnn import pad_sequence
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
DEVICE = torch.device("cuda:0")
MICRO_BATCH = 8
GRAD_ACCUM = 4
WARMUP_STEPS = 5
TIMED_STEPS = 20

set_seed(SEED)

tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    attn_implementation="sdpa",
).to(DEVICE)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
))

# Frozen FP16 base + FP32 adapters for mixed-precision training.
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.float()

# First measure without activation recomputation.
model.gradient_checkpointing_disable()

INITIAL_ADAPTER_DIR = OUTPUT_DIR / "initial_adapter_seed42"
model.save_pretrained(INITIAL_ADAPTER_DIR)
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer")
model.print_trainable_parameters()

trainable = {
    name: parameter
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}
assert all("lora_" in name for name in trainable)

original_adapters = {
    name: parameter.detach().cpu().clone()
    for name, parameter in trainable.items()
}

train_data = tokenized_ds["train"]
lengths = np.asarray(train_data["sequence_length"])
longest_indices = np.argsort(lengths)[-MICRO_BATCH:]
rng = np.random.default_rng(SEED)

# Reusable collator: right padding, preserving answer-only masks.
def training_collator(rows):
    padding_values = {
        "input_ids": tokenizer.pad_token_id,
        "attention_mask": 0,
        "labels": -100,
    }
    return {
        key: pad_sequence(
            [torch.tensor(row[key], dtype=torch.long) for row in rows],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding_values.items()
    }

optimizer = torch.optim.AdamW(
    trainable.values(), lr=1e-4, weight_decay=0.01
)
scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)
step_times, step_losses = [], []
skipped_warmup = 0
skipped_timed = 0
peak_gib = 0.0
batch = loss = None

try:
    model.train()
    torch.cuda.reset_peak_memory_stats(DEVICE)

    for step in range(WARMUP_STEPS + TIMED_STEPS):
        torch.cuda.synchronize(DEVICE)
        started = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)

        cpu_batches = []
        for micro in range(GRAD_ACCUM):
            # Stress-test the longest sequences during warmup.
            indices = (
                longest_indices
                if step == 0 and micro == 0
                else rng.choice(
                    len(train_data), MICRO_BATCH, replace=False
                )
            )
            cpu_batches.append(training_collator([
                train_data[int(index)] for index in indices
            ]))

        token_counts = [
            int((item["labels"][:, 1:] != -100).sum())
            for item in cpu_batches
        ]
        total_tokens = sum(token_counts)
        accumulated_loss = 0.0

        for cpu_batch, count in zip(cpu_batches, token_counts):
            batch = {
                key: value.to(DEVICE)
                for key, value in cpu_batch.items()
            }

            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss
                weighted_loss = loss * (count / total_tokens)

            assert torch.isfinite(loss), "Non-finite training loss."
            scaler.scale(weighted_loss).backward()
            accumulated_loss += loss.detach().item() * count / total_tokens

            del weighted_loss
            batch = loss = None

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            list(trainable.values()), max_norm=1.0
        )
        previous_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        skipped = int(scaler.get_scale() < previous_scale)

        torch.cuda.synchronize(DEVICE)
        elapsed = time.perf_counter() - started

        if step < WARMUP_STEPS:
            skipped_warmup += skipped
        else:
            skipped_timed += skipped
            step_times.append(elapsed)
            step_losses.append(accumulated_loss)

        if (step + 1) % 5 == 0:
            print(
                f"Step {step + 1}/{WARMUP_STEPS + TIMED_STEPS} | "
                f"{elapsed:.2f} seconds | loss={accumulated_loss:.4f}"
            )

    peak_gib = torch.cuda.max_memory_allocated(DEVICE) / 1024**3
    mean_seconds = float(np.mean(step_times))
    updates_per_epoch = math.ceil(
        len(train_data) / (MICRO_BATCH * GRAD_ACCUM)
    )

    benchmark = {
        "model": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "device": str(DEVICE),
        "micro_batch_size": MICRO_BATCH,
        "gradient_accumulation": GRAD_ACCUM,
        "effective_batch_size": MICRO_BATCH * GRAD_ACCUM,
        "gradient_checkpointing": False,
        "learning_rate": 1e-4,
        "warmup_steps": WARMUP_STEPS,
        "timed_steps": TIMED_STEPS,
        "mean_seconds_per_optimizer_step": mean_seconds,
        "estimated_steps_per_epoch": updates_per_epoch,
        "estimated_training_minutes_per_epoch":
            updates_per_epoch * mean_seconds / 60,
        "peak_allocated_gib_including_longest_batch": peak_gib,
        "skipped_warmup_steps": skipped_warmup,
        "skipped_timed_steps": skipped_timed,
        "note": "Estimate excludes validation and checkpoint saving.",
    }

    (OUTPUT_DIR / "training_benchmark.json").write_text(
        json.dumps(benchmark, indent=2)
    )
    print("\nBENCHMARK")
    print(json.dumps(benchmark, indent=2))

finally:
    with torch.no_grad():
        for name, parameter in trainable.items():
            parameter.copy_(original_adapters[name])

    model.zero_grad(set_to_none=True)
    model.eval()
    batch = loss = None
    del optimizer, scaler, original_adapters
    gc.collect()
    torch.cuda.empty_cache()
    set_seed(SEED)

    print("\nOriginal adapter weights restored. Benchmark updates discarded.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
Step 5/25 | 1.61 seconds | loss=0.3627
Step 10/25 | 1.51 seconds | loss=0.3785
Step 15/25 | 1.63 seconds | loss=0.3489
Step 20/25 | 1.52 seconds | loss=0.3759
Step 25/25 | 1.91 seconds | loss=0.3342

BENCHMARK
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "model_revision": "7ae557604adf67be50417f59c2c2f167def9a775",
  "device": "cuda:0",
  "micro_batch_size": 8,
  "gradient_accumulation": 4,
  "effective_batch_size": 32,
  "gradient_checkpointing": false,
  "learning_rate": 0.0001,
  "warmup_steps": 5,
  "timed_steps": 20,
  "mean_seconds_per_optimizer_step": 1.6810325504500014,
  "estimated_steps_per_epoch": 331,
  "estimated_training_minutes_per_epoch": 9.273696236649176,
  "peak_allocated_gib_including_longest_batch": 9.865746974945068,
  "skipped_warmup_steps": 0,
  "skipped_timed_steps": 0,
  "note": "Estimate excludes validation and checkpoint saving."
}

Original adapter weights restored. Benchmark updat

In [8]:
import json
import time
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import GenerationConfig
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

ZERO_DIR = OUTPUT_DIR / "Qwen_ZeroShot_Validation"
ZERO_DIR.mkdir(parents=True, exist_ok=True)

EVAL_BATCH_SIZE = 8

eval_generation = GenerationConfig(
    max_new_tokens=12,
    do_sample=False,
    num_beams=1,
    use_cache=True,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

label_lookup = {label.lower(): label for label in LABELS}

def parse_prediction(text):
    normalized = " ".join(text.strip().lower().split())
    return label_lookup.get(normalized, "INVALID")

previous_padding = tokenizer.padding_side
previous_training = model.training
records = []
started = time.perf_counter()

try:
    tokenizer.padding_side = "left"
    model.eval()

    with model.disable_adapter(), torch.inference_mode(), (
        ZERO_DIR / "predictions.jsonl"
    ).open("w") as output:

        for offset in tqdm(
            range(0, len(raw_ds["validation"]), EVAL_BATCH_SIZE),
            desc="Zero-shot validation",
        ):
            rows = raw_ds["validation"][offset:offset + EVAL_BATCH_SIZE]

            prompts = [
                tokenizer.apply_chat_template(
                    make_messages(claim, evidence),
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for claim, evidence in zip(rows["claim"], rows["evidence"])
            ]

            inputs = tokenizer(
                prompts,
                padding=True,
                truncation=False,
                add_special_tokens=False,
                return_tensors="pt",
            ).to(DEVICE)

            width = inputs["input_ids"].shape[1]
            assert width + eval_generation.max_new_tokens <= MAX_SEQUENCE_LENGTH

            generated = model.generate(
                **inputs,
                generation_config=eval_generation,
            )
            responses = tokenizer.batch_decode(
                generated[:, width:],
                skip_special_tokens=True,
            )

            for uid, gold, response in zip(
                rows["id"], rows["label"], responses
            ):
                record = {
                    "id": uid,
                    "true_label": gold,
                    "predicted_label": parse_prediction(response),
                    "raw_response": response,
                }
                records.append(record)
                output.write(json.dumps(record) + "\n")

            del inputs, generated

finally:
    tokenizer.padding_side = previous_padding
    model.train(previous_training)
    torch.cuda.empty_cache()

predictions = pd.DataFrame(records)
assert len(predictions) == len(raw_ds["validation"])

gold = predictions["true_label"]
predicted = predictions["predicted_label"]

metrics = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "evaluation_split": "validation",
    "n_examples": len(predictions),
    "adapters_enabled": False,
    "accuracy": float(accuracy_score(gold, predicted)),
    "macro_f1": float(f1_score(
        gold, predicted, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "weighted_f1": float(f1_score(
        gold, predicted, labels=LABELS,
        average="weighted", zero_division=0,
    )),
    "invalid_rate": float((predicted == "INVALID").mean()),
    "elapsed_minutes": (time.perf_counter() - started) / 60,
}

report = classification_report(
    gold, predicted, labels=LABELS,
    output_dict=True, zero_division=0,
)

matrix_labels = LABELS + ["INVALID"]
matrix = pd.DataFrame(
    confusion_matrix(gold, predicted, labels=matrix_labels),
    index=matrix_labels,
    columns=matrix_labels,
).loc[LABELS]

(ZERO_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(ZERO_DIR / "classification_report.json").write_text(
    json.dumps(report, indent=2)
)
matrix.to_csv(ZERO_DIR / "confusion_matrix.csv")
eval_generation.save_pretrained(ZERO_DIR)

print("\nZERO-SHOT VALIDATION")
print(json.dumps(metrics, indent=2))
print(classification_report(
    gold, predicted, labels=LABELS, digits=4, zero_division=0,
))
print("CONFUSION MATRIX — rows: actual; columns: predicted")
display(matrix)
print("Saved:", ZERO_DIR)

Zero-shot validation:   0%|          | 0/240 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



ZERO-SHOT VALIDATION
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "model_revision": "7ae557604adf67be50417f59c2c2f167def9a775",
  "dataset_revision": "b20ac99ceed62f5264a319fa25a854df1668d85b",
  "evaluation_split": "validation",
  "n_examples": 1917,
  "adapters_enabled": false,
  "accuracy": 0.5362545644235786,
  "macro_f1": 0.39414891172494143,
  "weighted_f1": 0.47689002579925494,
  "invalid_rate": 0.0,
  "elapsed_minutes": 0.6146522620999993
}
              precision    recall  f1-score   support

    Supports     0.6527    0.2045    0.3114       533
     Refutes     0.2893    0.1458    0.1939       391
     Neutral     0.5551    0.8681    0.6771       993

    accuracy                         0.5363      1917
   macro avg     0.4990    0.4061    0.3941      1917
weighted avg     0.5280    0.5363    0.4769      1917

CONFUSION MATRIX — rows: actual; columns: predicted


,Supports,Refutes,Neutral,INVALID
Supports,109,52,372,0
Refutes,15,57,319,0
Neutral,43,88,862,0


Saved: /kaggle/working/HealthVer_Experiments/Qwen_ZeroShot_Validation


In [9]:
import gc
import json
import math
import time

import numpy as np
import torch

from transformers import Trainer, TrainingArguments, set_seed
from transformers.trainer_utils import get_last_checkpoint
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
)

RUN_DIR = OUTPUT_DIR / "Qwen_LoRA_full_seed42"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Keep Trainer on one GPU despite two GPUs being visible.
# This prevents automatic DataParallel and preserves the benchmark batch size.
class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

# Evaluate generated answers without supplying gold labels to the model.
class HealthVerTrainer(Trainer):
    def evaluate(
        self, eval_dataset=None, ignore_keys=None,
        metric_key_prefix="eval", **kwargs
    ):
        previous_training = self.model.training
        previous_padding = tokenizer.padding_side
        records = []
        started = time.perf_counter()

        prediction_path = (
            RUN_DIR / f"validation_step_{self.state.global_step}.jsonl"
        )

        try:
            self.model.eval()
            tokenizer.padding_side = "left"

            with torch.inference_mode(), prediction_path.open("w") as output:
                for offset in range(
                    0, len(raw_ds["validation"]), EVAL_BATCH_SIZE
                ):
                    rows = raw_ds["validation"][
                        offset:offset + EVAL_BATCH_SIZE
                    ]

                    prompts = [
                        tokenizer.apply_chat_template(
                            make_messages(claim, evidence),
                            tokenize=False,
                            add_generation_prompt=True,
                        )
                        for claim, evidence in zip(
                            rows["claim"], rows["evidence"]
                        )
                    ]

                    inputs = tokenizer(
                        prompts,
                        padding=True,
                        truncation=False,
                        add_special_tokens=False,
                        return_tensors="pt",
                    ).to(DEVICE)

                    width = inputs["input_ids"].shape[1]
                    assert (
                        width + eval_generation.max_new_tokens
                        <= MAX_SEQUENCE_LENGTH
                    )

                    # LoRA adapters stay enabled.
                    generated = self.model.generate(
                        **inputs,
                        generation_config=eval_generation,
                    )
                    responses = tokenizer.batch_decode(
                        generated[:, width:],
                        skip_special_tokens=True,
                    )

                    for uid, gold, response in zip(
                        rows["id"], rows["label"], responses
                    ):
                        record = {
                            "id": uid,
                            "true_label": gold,
                            "predicted_label": parse_prediction(response),
                            "raw_response": response,
                        }
                        records.append(record)
                        output.write(json.dumps(record) + "\n")

                    del inputs, generated

            assert len(records) == len(raw_ds["validation"])
            gold = [row["true_label"] for row in records]
            predicted = [row["predicted_label"] for row in records]

            metrics = {
                f"{metric_key_prefix}_accuracy":
                    float(accuracy_score(gold, predicted)),
                f"{metric_key_prefix}_macro_f1":
                    float(f1_score(
                        gold, predicted, labels=LABELS,
                        average="macro", zero_division=0,
                    )),
                f"{metric_key_prefix}_weighted_f1":
                    float(f1_score(
                        gold, predicted, labels=LABELS,
                        average="weighted", zero_division=0,
                    )),
                f"{metric_key_prefix}_invalid_rate":
                    float(np.mean(np.asarray(predicted) == "INVALID")),
                f"{metric_key_prefix}_runtime":
                    time.perf_counter() - started,
            }

            report = classification_report(
                gold, predicted, labels=LABELS,
                output_dict=True, zero_division=0,
            )
            prediction_path.with_suffix(".metrics.json").write_text(
                json.dumps({
                    "metrics": metrics,
                    "classification_report": report,
                }, indent=2)
            )

            print("\nVALIDATION")
            print(classification_report(
                gold, predicted, labels=LABELS,
                digits=4, zero_division=0,
            ))

            self.log(metrics)
            self.control = self.callback_handler.on_evaluate(
                self.args, self.state, self.control, metrics
            )
            return metrics

        finally:
            tokenizer.padding_side = previous_padding
            self.model.train(previous_training)
            gc.collect()
            with torch.cuda.device(DEVICE):
                torch.cuda.empty_cache()

            print(
                f"GPU 0 allocated: "
                f"{torch.cuda.memory_allocated(DEVICE)/1024**3:.2f} GiB"
            )

EPOCHS = 3
updates_per_epoch = math.ceil(
    len(tokenized_ds["train"]) / (MICRO_BATCH * GRAD_ACCUM)
)

args = SingleGPUArguments(
    output_dir=str(RUN_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=1e-4,
    warmup_steps=math.ceil(0.03 * EPOCHS * updates_per_epoch),
    lr_scheduler_type="linear",
    optim="adamw_torch",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=False,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=25,
    logging_first_step=True,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    seed=SEED,
    data_seed=SEED,
    label_names=["labels"],
)

assert args.world_size == 1
assert args.device.type == "cuda" and args.device.index == 0
assert args.train_batch_size == MICRO_BATCH

run_config = {
    "model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_revision": DATASET_REVISION,
    "training_rows": len(tokenized_ds["train"]),
    "validation_rows": len(raw_ds["validation"]),
    "epochs": EPOCHS,
    "micro_batch_size": MICRO_BATCH,
    "gradient_accumulation": GRAD_ACCUM,
    "learning_rate": 1e-4,
    "seed": SEED,
    "sequence_limit": MAX_SEQUENCE_LENGTH,
    "selection_metric": "validation_macro_f1",
}

config_path = RUN_DIR / "run_config.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == run_config, (
        "Settings differ from the existing run. Use a new run directory."
    )
else:
    config_path.write_text(json.dumps(run_config, indent=2))

set_seed(SEED)
tokenizer.padding_side = "right"
model.config.use_cache = False
model.gradient_checkpointing_disable()

trainer = HealthVerTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"].select_columns(
        ["input_ids", "attention_mask", "labels"]
    ),
    eval_dataset=tokenized_ds["validation"],
    data_collator=training_collator,
    processing_class=tokenizer,
)

checkpoint = get_last_checkpoint(str(RUN_DIR))

print("Device:", args.device)
print("Training GPUs:", args.n_gpu)
print("Effective batch:", args.train_batch_size * GRAD_ACCUM)
print("Expected optimizer steps:", EPOCHS * updates_per_epoch)
print("Resuming:", checkpoint or "fresh run")

result = trainer.train(resume_from_checkpoint=checkpoint)

# Best validation-macro-F1 checkpoint is now loaded.
trainer.save_model(str(RUN_DIR / "best_adapter"))
tokenizer.save_pretrained(RUN_DIR / "best_adapter")
trainer.save_metrics("train", result.metrics)
trainer.save_state()

print("\nBest checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation macro-F1:", trainer.state.best_metric)
print("Best adapter saved:", RUN_DIR / "best_adapter")
print("Test set remains unevaluated.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Device: cuda:0
Training GPUs: 1
Effective batch: 32
Expected optimizer steps: 993
Resuming: fresh run


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Rate
1,0.125031,No log,0.796035,0.762532,0.795635,0.000000
2,0.076525,No log,0.820553,0.797225,0.823264,0.000000
3,0.031480,No log,0.825248,0.801597,0.827953,0.000000



VALIDATION
              precision    recall  f1-score   support

    Supports     0.7634    0.6660    0.7114       533
     Refutes     0.6539    0.7442    0.6962       391
     Neutral     0.8739    0.8862    0.8800       993

    accuracy                         0.7960      1917
   macro avg     0.7638    0.7655    0.7625      1917
weighted avg     0.7983    0.7960    0.7956      1917

GPU 0 allocated: 1.04 GiB

VALIDATION
              precision    recall  f1-score   support

    Supports     0.6923    0.8780    0.7742       533
     Refutes     0.7690    0.6982    0.7319       391
     Neutral     0.9391    0.8379    0.8856       993

    accuracy                         0.8206      1917
   macro avg     0.8001    0.8047    0.7972      1917
weighted avg     0.8358    0.8206    0.8233      1917

GPU 0 allocated: 1.04 GiB

VALIDATION
              precision    recall  f1-score   support

    Supports     0.7657    0.7786    0.7721       533
     Refutes     0.6903    0.7980    0.74